# Step 3 — TOD Station Assignment Re-integration

Reads the manually-reviewed conflict and orphan resolution layers from the shared
GeoPackage, applies valid assignments back to the main stop and access-point
datasets, and re-exports the corrected `tod_stops` and `tod_access_points` layers.

> **Pipeline order:** Run after completing the manual GIS review that follows
> `2_tod_stop_and_access_assignment.ipynb`.
> - Open `tod_stops_review_v1` and `tod_access_review_v1` in GIS.
> - For each conflict row, set `valid_station_assignment = 1` on the correct row.
> - For no-match orphans, fill in `station_id` and set `valid_station_assignment = 1`.
> - Save the layers back to the GeoPackage, then run this notebook.
>
> All data source paths are defined in `config.py`.


In [ ]:
import pandas as pd
import geopandas as gpd
from config import *

In [ ]:
# All paths and layer names are imported from config.py
tod_gtfs_db_gdb_path = TOD_DATABASE_GPKG

In [ ]:
# Read back manually-reviewed layers (conflicts + orphans combined).
# These are saved by the reviewer in GIS as GPKG_TOD_STOPS_REVIEWED_LAYER
# and GPKG_TOD_ACCESS_REVIEWED_LAYER after editing tod_stops_review /
# tod_access_review.  See config.py for the expected layer names.
stops_review_reviewed = gpd.read_file(tod_gtfs_db_gdb_path, layer=GPKG_TOD_STOPS_REVIEWED_LAYER)
access_pts_review_reviewed = gpd.read_file(tod_gtfs_db_gdb_path, layer=GPKG_TOD_ACCESS_REVIEWED_LAYER)

print(f"Loaded {len(stops_review_reviewed)} stop review rows")
print(f"Loaded {len(access_pts_review_reviewed)} access point review rows")

# Filter to valid assignments only (one row per stop/access point with correct station_id)
stops_valid = stops_review_reviewed[stops_review_reviewed["valid_station_assignment"] == 1][
    ["stop_id", "station_id"]
]
access_pts_valid = access_pts_review_reviewed[
    access_pts_review_reviewed["valid_station_assignment"] == 1
][["access_id", "station_id"]]

print(f"\nValid stop assignments from review: {len(stops_valid)}")
print(f"Valid access point assignments from review: {len(access_pts_valid)}")

# Warn if any point has more than one valid assignment (data issue)
dup_stops = stops_valid["stop_id"].duplicated().sum()
dup_access = access_pts_valid["access_id"].duplicated().sum()
if dup_stops > 0:
    print(
        f"⚠️  {dup_stops} stop_ids have multiple valid_station_assignment=1 rows — check your data!"
    )
if dup_access > 0:
    print(
        f"⚠️  {dup_access} access_ids have multiple valid_station_assignment=1 rows — check your data!"
    )

In [ ]:
# Read assignment outputs written by notebook 2
stops_final = gpd.read_file(tod_gtfs_db_gdb_path, layer=GPKG_TOD_STOPS_LAYER)
access_pts_final = gpd.read_file(tod_gtfs_db_gdb_path, layer=GPKG_TOD_ACCESS_PTS_LAYER)

print(f"Loaded {len(stops_final)} TOD stops from GeoPackage")
print(f"Loaded {len(access_pts_final)} access points from GeoPackage")

# Apply reviewed conflict assignments back to the main datasets
stops_final_v2 = stops_final.copy()
access_pts_final_v2 = access_pts_final.copy()

# Update stops: set station_id from valid reviewed assignments
if len(stops_valid) > 0:
    stops_final_v2 = stops_final_v2.set_index("stop_id")
    stops_final_v2.loc[stops_final_v2.index.isin(stops_valid["stop_id"]), "station_id"] = (
        stops_valid.set_index("stop_id")["station_id"]
    )
    stops_final_v2 = stops_final_v2.reset_index()

# Update access points: set station_id from valid reviewed assignments
if len(access_pts_valid) > 0:
    access_pts_final_v2 = access_pts_final_v2.set_index("access_id")
    access_pts_final_v2.loc[
        access_pts_final_v2.index.isin(access_pts_valid["access_id"]), "station_id"
    ] = access_pts_valid.set_index("access_id")["station_id"]
    access_pts_final_v2 = access_pts_final_v2.reset_index()

# Report final assignment coverage
print("\n=== POST-REVIEW ASSIGNMENT SUMMARY ===")
print(f"TOD stops assigned: {stops_final_v2['station_id'].notna().sum()} / {len(stops_final_v2)}")
print(f"  Still unassigned: {stops_final_v2['station_id'].isna().sum()}")
print(
    f"\nAccess points assigned: {access_pts_final_v2['station_id'].notna().sum()} / {len(access_pts_final_v2)}"
)
print(f"  Still unassigned: {access_pts_final_v2['station_id'].isna().sum()}")

In [ ]:
# Re-export tod_stops and tod_access_points with final corrected datasets.
stops_final_v2.to_file(tod_gtfs_db_gdb_path, layer=GPKG_TOD_STOPS_LAYER, driver="GPKG", mode="w")
print(f"✅ Exported {len(stops_final_v2)} TOD stops (with conflict resolutions applied)")

access_pts_final_v2.to_file(
    tod_gtfs_db_gdb_path, layer=GPKG_TOD_ACCESS_PTS_LAYER, driver="GPKG", mode="w"
)
print(f"✅ Exported {len(access_pts_final_v2)} access points (with conflict resolutions applied)")

print(f"\n📁 Final output: {tod_gtfs_db_gdb_path}")